In [32]:
from pathlib import Path

import json
import polars as pl

In [33]:
# ricu concept dict
ricu_cnpt_dict_df = pl.read_json("concept-dict.json")
# miiv data
miiv_d_items_lf = pl.scan_csv("/workspaces/example/data/physionet.org/files/mimiciv/3.1/icu/d_items.csv.gz").with_columns((pl.col("itemid").cast(pl.Utf8) + "//" + pl.col("label")).alias("code"))
miiv_d_labitems_lf = pl.scan_csv("/workspaces/example/data/physionet.org/files/mimiciv/3.1/hosp/d_labitems.csv.gz").with_columns((pl.col("itemid").cast(pl.Utf8) + "//" + pl.col("label")).alias("code"))

# Function
creates two dicts with usable and (currently) unusable concept informations and prepares the usable information

In [34]:
from typing import Any
import polars as pl


def normalize_label(value: str) -> str:
    return value.replace(" ", "_").replace("/", "_per_")


def get_concept_entry(info: pl.DataFrame, concept_key: str) -> dict[str, Any]:
    return info[concept_key][0]


def get_sources_for_dataset(
    concept_entry: dict[str, Any],
    src_name: str,
) -> list[dict[str, Any]]:
    sources = concept_entry.get("sources")
    if not sources:
        return []
    return sources.get(src_name, [])


def is_string_id_source(ids: Any) -> bool:
    if isinstance(ids, str):
        return True
    if isinstance(ids, list) and ids and isinstance(ids[0], str):
        return True
    return False


def normalize_ids(ids: Any) -> list[int] | None:
    if ids is None:
        return None
    if isinstance(ids, list):
        return ids
    return [ids]


def build_code_regex(
    table_name: str,
    ids: list[int] | None,
    src_regex: str | None,
    src_items: dict[str, pl.LazyFrame],
    src_item_col_name: str,
) -> str | None:
    if ids is None:
        return src_regex

    lookup_table = table_name if table_name in src_items else "else"
    filtered = (
        src_items[lookup_table]
        .filter(pl.col(src_item_col_name).is_in(ids))
        .select("code")
        .collect()
    )

    codes = filtered["code"].to_list()
    regex = "|".join(codes)
    return f"({regex})"


def build_output_entry(
    concept_short: str,
    concept_entry: dict[str, Any],
    source_entry: dict[str, Any],
    normalized_name: str,
    normalized_ids: list[int] | None,
    code_regex: str | None,
) -> dict[str, Any]:
    return {
        "name": normalized_name,
        "unit": concept_entry.get("unit"),
        "table": source_entry["table"],
        "code": code_regex,
        "ids": normalized_ids,
        "src_regex": source_entry.get("regex"),
        "short": concept_short,
        "min": concept_entry.get("min", "-Inf") if concept_entry.get("min") is not None else "-Inf",
        "max": concept_entry.get("max", "Inf") if concept_entry.get("max") is not None else "Inf",
    }


def should_store_as_unhandled(source_entry: dict[str, Any]) -> bool:
    return source_entry.get("ids") is None and source_entry.get("regex") is None


def should_skip_to_unhandled(ids: Any) -> bool:
    return is_string_id_source(ids)


def add_entry_to_data(
    data: dict[str, dict[str, list[dict[str, Any]]]],
    category: str,
    name: str,
    entry: dict[str, Any],
) -> None:
    data.setdefault(category, {}).setdefault(name, []).append(entry)


def ricu_cnpt_dict_to_cust_cnpt_dict(
    info: pl.DataFrame,
    src_name: str,
    src_items: dict[str, pl.LazyFrame],
    src_item_col_name: str,
) -> tuple[dict, dict]:
    data: dict[str, dict[str, list[dict[str, Any]]]] = {}
    un_src_data: dict[str, Any] = {}
    
    for concept_key in info.columns:
        concept_entry = get_concept_entry(info, concept_key)
        source_entries = get_sources_for_dataset(concept_entry, src_name)

        if concept_entry.get("class"):
            continue

        if not source_entries:
            continue

        for source_entry in source_entries:
            if should_store_as_unhandled(source_entry):
                un_src_data[concept_key] = info[concept_key]
                continue

            raw_ids = source_entry.get("ids")
            src_regex = source_entry.get("regex")

            if should_skip_to_unhandled(raw_ids):
                un_src_data[concept_key] = info[concept_key]
                continue

            category = normalize_label(concept_entry["category"])
            name = normalize_label(concept_entry["description"])
            table_name = source_entry["table"]
            normalized_ids = normalize_ids(raw_ids)

            code_regex = build_code_regex(
                table_name=table_name,
                ids=normalized_ids,
                src_regex=src_regex,
                src_items=src_items,
                src_item_col_name=src_item_col_name,
            )

            output_entry = build_output_entry(
                concept_short=concept_key,
                concept_entry=concept_entry,
                source_entry=source_entry,
                normalized_name=name,
                normalized_ids=normalized_ids,
                code_regex=code_regex,
            )

            add_entry_to_data(
                data=data,
                category=category,
                name=name,
                entry=output_entry,
            )

    return data, un_src_data

In [35]:
data_by_cat = {}
un_src_data = {}

src_item_mapping = {"labevents": miiv_d_labitems_lf, "else": miiv_d_items_lf}
(data_by_cat, un_src_data) = ricu_cnpt_dict_to_cust_cnpt_dict(ricu_cnpt_dict_df, "miiv", src_item_mapping, "itemid")

In [36]:
data_by_cat

{'hematology': {'mean_corpuscular_hemoglobin_concentration': [{'name': 'mean_corpuscular_hemoglobin_concentration',
    'unit': '%',
    'table': 'labevents',
    'code': '(51249//MCHC)',
    'ids': [51249],
    'src_regex': None,
    'short': 'mchc',
    'min': 20,
    'max': 50}],
  'fibrinogen': [{'name': 'fibrinogen',
    'unit': 'mg/dL',
    'table': 'labevents',
    'code': '(51214//Fibrinogen, Functional)',
    'ids': [51214],
    'src_regex': None,
    'short': 'fgn',
    'min': 0,
    'max': 1500}],
  'prothrombine_time': [{'name': 'prothrombine_time',
    'unit': 'sec',
    'table': 'labevents',
    'code': '(51274//PT)',
    'ids': [51274],
    'src_regex': None,
    'short': 'pt',
    'min': 0,
    'max': 'Inf'}],
  'Hemoglobin_A1C': [{'name': 'Hemoglobin_A1C',
    'unit': '%',
    'table': 'labevents',
    'code': '(50852//% Hemoglobin A1c)',
    'ids': [50852],
    'src_regex': None,
    'short': 'hba1c',
    'min': '-Inf',
    'max': 'Inf'}],
  'erythrocyte_distribution_